# Estudo comparativo: CNN vs Vision Transformer na detecção de imagens Real vs IA

**2ª Avaliação — Análise comparativa com 30 simulações e teste de hipótese**

| | Técnica 1 (fixa) | Técnica 2 (contrafactual) | Técnica 3 (fixa) |
|---|---|---|---|
| **Cenário A (H0)** | Transfer Learning (ImageNet) | CNN — ResNet-50 | xAI — Grad-CAM |
| **Cenário B (H1)** | Transfer Learning (ImageNet) | Transformer — ViT-S/16 | xAI — Grad-CAM |

- Dataset: [cashbowman/ai-generated-images-vs-real-images](https://www.kaggle.com/datasets/cashbowman/ai-generated-images-vs-real-images)
- 30 simulações pareadas (o mesmo split estratificado 70/15/15 por seed para os dois modelos)
- H0: μ(F1_ViT − F1_CNN) = 0 · H1: ≠ 0 · α = 0,05 · Shapiro-Wilk → t pareado ou Wilcoxon

> **Antes de rodar:** *Ambiente de execução → Alterar o tipo de ambiente → GPU T4*.
> Os resultados vão para o Google Drive, então se o Colab cair é só rodar de novo: as simulações já feitas são puladas.

## 1. Ambiente

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
REPO_URL = "https://github.com/Breno-Lira/Plant-seedlings.git"
!git clone -q $REPO_URL /content/repo 2>/dev/null || git -C /content/repo pull -q
%cd /content/repo
!pip install -q timm kagglehub grad-cam

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
RESULTS = "/content/drive/MyDrive/estudo_comparativo/results"
!mkdir -p $RESULTS

## 2. Credenciais do Kaggle (opcional)
O dataset é público e o `kagglehub` baixa sem login (testado). Se o download falhar
(ex.: limite de requisições), adicione em **🔑 Secrets** as chaves `KAGGLE_USERNAME` e
`KAGGLE_KEY` (kaggle.com → Settings → API → *Create New Token*) e rode esta célula.

In [ ]:
import os
from google.colab import userdata
for k in ("KAGGLE_USERNAME", "KAGGLE_KEY"):
    try:
        os.environ[k] = userdata.get(k)
    except Exception as e:
        print(f"{k} não configurado — usando download anônimo")

## 3. Análise exploratória
Baixa o dataset, descarta arquivos corrompidos, remove duplicatas exatas (md5) e
salva uma cópia padronizada (RGB, menor lado = 256 px, PNG) para que formato e
resolução originais não sirvam de "atalho" para o classificador.

In [ ]:
!python -m src.eda --results-dir $RESULTS

In [ ]:
from IPython.display import Image, display
for f in ["fig_eda_formats", "fig_eda_samples"]:
    display(Image(f"{RESULTS}/figures/{f}.png", width=800))

## 4. Smoke test (≈2 min)
Duas simulações com 1 época, numa pasta separada, só para garantir que tudo roda.

In [ ]:
!python -m src.run_simulations --results-dir /content/smoke --n-sim 2 --epochs 1 --quiet
!python -m src.stats --results-dir /content/smoke

## 5. As 30 simulações (≈1,5–2 h na T4)
Para cada seed: split estratificado → ResNet-50 e ViT-S/16 treinados com o mesmo protocolo
(AdamW 1e-4, cosine, 10 épocas, early stopping paciência 3 em F1 de validação, AMP) →
métricas no teste. Resultados em `simulations.csv`, histórico em `history.csv`,
probabilidades em `preds/`.

In [ ]:
!python -m src.run_simulations --results-dir $RESULTS --n-sim 30 --epochs 10 --quiet

## 6. Teste de hipótese

In [ ]:
!python -m src.stats --results-dir $RESULTS

In [ ]:
import json, pandas as pd
s = json.load(open(f"{RESULTS}/stats.json"))
pd.DataFrame({m: s[m] for m in ["f1_macro", "accuracy", "auc"]}).drop(index=["ci95_diff"], errors="ignore")

In [ ]:
for f in ["fig_boxplot_metrics", "fig_paired_f1", "fig_diff_hist_qq",
          "fig_confusion_mean", "fig_roc_mean", "fig_training_curves"]:
    display(Image(f"{RESULTS}/figures/{f}.png", width=800))

## 7. Explicabilidade (xAI)
Retreina os dois modelos no seed representativo (ΔF1 mais próximo da mediana) e gera
Grad-CAM + curva de *deletion* (fidelidade das explicações).

In [ ]:
!python -m src.xai --results-dir $RESULTS

In [ ]:
for f in ["fig_gradcam_grid", "fig_deletion_curves"]:
    display(Image(f"{RESULTS}/figures/{f}.png", width=900))

## 8. Exportar resultados
Baixe o zip e extraia na pasta `results/` do repositório (o cache de imagens fica de fora).

In [ ]:
!cd $RESULTS/.. && zip -qr /content/results.zip results -x "results/cache/*"
from google.colab import files
files.download("/content/results.zip")